In [1]:
# ============================================================
# STEP 4 - EXPERIMENT 1
# Full Balanced Score Ranking Table
#
# Input expected in the same working folder:
#   - experiment_input.zip
#   - plan.md                         optional, copied into metadata if available
#   - sp500_constituents_2020_start.csv optional, not used in Experiment 1
#
# Output folder:
#   - tda_step4_exp1_outputs/
#
# Output zip:
#   - tda_step4_exp1_outputs.zip
#
# Required outputs generated:
#   - step4_full_balanced_score_ranking.csv
#   - step4_full_balanced_score_ranking.md
#   - step4_balanced_score_formula.json
#   - step4_top_configurations.csv
#   - step4_balanced_score_audit.md
# ============================================================

import os
import re
import json
import shutil
import zipfile
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


# ============================================================
# 0. USER SETTINGS
# ============================================================

INPUT_ZIP = Path("experiment_input.zip")
PLAN_MD = Path("plan.md")
SP500_2020_FILE = Path("sp500_constituents_2020_start.csv")  # not used in Exp 1, but logged if present

OUTPUT_ROOT = Path("tda_step4_exp1_outputs")
WORK_ROOT = Path("_step4_exp1_work")

CORE_METHODS = ["Mapper", "Global DBSCAN", "PCA-KMeans"]
BENCHMARK_METHODS = ["Equal Weight Universe", "S&P 500"]
REBALANCE_FREQUENCIES = [21, 42, 63]
SELECTION_FRACTIONS = [0.20, 0.30, 0.40]

BALANCED_SCORE_WEIGHTS = {
    "Annualized Return Score": 0.35,
    "Daily Sharpe Score": 0.35,
    "Daily Max Drawdown Score": 0.20,
    "Average Turnover Score": 0.10,
}

METRIC_COLUMNS = [
    "Cumulative Return",
    "Annualized Return",
    "Daily Sharpe",
    "Daily Max Drawdown",
    "Average Turnover",
    "Average Number of Stocks",
]

REQUIRED_OUTPUTS = [
    "step4_full_balanced_score_ranking.csv",
    "step4_full_balanced_score_ranking.md",
    "step4_balanced_score_formula.json",
    "step4_top_configurations.csv",
    "step4_balanced_score_audit.md",
]


# ============================================================
# 1. HELPER FUNCTIONS
# ============================================================

def sha256_file(path: Path) -> str:
    if not path.exists():
        return ""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def clean_folder(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def extract_zip(zip_path: Path, dest: Path):
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest)


def extract_all_nested_zips(root: Path):
    nested_zip_paths = list(root.rglob("*.zip"))
    extracted_dirs = []
    for zp in nested_zip_paths:
        target = zp.parent / f"__extracted__{zp.stem}"
        if target.exists():
            shutil.rmtree(target)
        target.mkdir(parents=True, exist_ok=True)
        extract_zip(zp, target)
        extracted_dirs.append(target)
    return extracted_dirs


def find_first_file(root: Path, filename: str):
    matches = list(root.rglob(filename))
    if not matches:
        return None
    return matches[0]


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    unnamed = [c for c in df.columns if c.startswith("Unnamed")]
    if unnamed:
        df = df.drop(columns=unnamed)
    return df


def make_strategy_label(row):
    method = row.get("Method")
    h = row.get("Rebalance Frequency")
    l = row.get("l")
    if pd.notna(l):
        return f"{method}, h={int(h)}, l={int(round(float(l) * 100))}%"
    return f"{method}, h={int(h)}"


def coerce_numeric(df: pd.DataFrame, cols) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def normalized_rank_score(series: pd.Series, higher_is_better: bool) -> pd.Series:
    """
    Converts rank into a normalized score in (0, 1], where best = 1.
    With N rows:
      best rank 1 -> N/N = 1
      worst rank N -> 1/N
    Ties use min rank, matching the previous Step 3 convention closely.
    """
    n = series.notna().sum()
    ranks = series.rank(
        ascending=not higher_is_better,
        method="min",
        na_option="bottom"
    )
    return (n - ranks + 1) / n


def dataframe_to_markdown(df: pd.DataFrame, float_digits: int = 6) -> str:
    """
    Simple markdown table writer that does not require tabulate.
    """
    display_df = df.copy()

    for col in display_df.columns:
        if pd.api.types.is_float_dtype(display_df[col]):
            display_df[col] = display_df[col].map(
                lambda x: "" if pd.isna(x) else f"{x:.{float_digits}f}"
            )
        else:
            display_df[col] = display_df[col].map(
                lambda x: "" if pd.isna(x) else str(x)
            )

    cols = list(display_df.columns)
    header = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join(["---"] * len(cols)) + " |"

    rows = []
    for _, r in display_df.iterrows():
        rows.append("| " + " | ".join(str(r[c]) for c in cols) + " |")

    return "\n".join([header, sep] + rows)


def write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)


def safe_round_for_output(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            df[col] = df[col].round(10)
    return df


# ============================================================
# 2. PREPARE FOLDERS AND EXTRACT INPUTS
# ============================================================

if not INPUT_ZIP.exists():
    raise FileNotFoundError(
        f"Cannot find {INPUT_ZIP.resolve()}. "
        "Please place experiment_input.zip in the same folder as this notebook."
    )

clean_folder(WORK_ROOT)
clean_folder(OUTPUT_ROOT)

EXTRACT_ROOT = WORK_ROOT / "experiment_input_extracted"
extract_zip(INPUT_ZIP, EXTRACT_ROOT)
nested_dirs = extract_all_nested_zips(EXTRACT_ROOT)

metadata_dir = OUTPUT_ROOT / "00_metadata"
tables_dir = OUTPUT_ROOT / "01_full_ranking"
metadata_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

if PLAN_MD.exists():
    shutil.copy2(PLAN_MD, metadata_dir / "experiment_plan.md")

if SP500_2020_FILE.exists():
    shutil.copy2(SP500_2020_FILE, metadata_dir / "sp500_constituents_2020_start__not_used_in_exp1.csv")


# ============================================================
# 3. LOAD THE BEST AVAILABLE CONFIGURATION METRICS
# ============================================================

# Preferred source:
#   Step 3 already contains all 33 scored rows:
#   27 core configurations + 6 benchmark rows.
#
# We still recompute all score columns below instead of trusting Step 3's saved scores.

step3_all_path = find_first_file(EXTRACT_ROOT, "step3_all_configurations_scored.csv")

if step3_all_path is None:
    raise FileNotFoundError(
        "Could not find step3_all_configurations_scored.csv inside experiment_input.zip. "
        "Experiment 1 needs the full 27-row core grid plus benchmark rows. "
        "Please check that tda_experiment_outputs_step3.zip is included."
    )

raw = pd.read_csv(step3_all_path)
raw = normalize_columns(raw)

required_base_cols = [
    "Method",
    "Rebalance Frequency",
    "l",
    "Cumulative Return",
    "Annualized Return",
    "Daily Sharpe",
    "Daily Max Drawdown",
    "Average Turnover",
    "Average Number of Stocks",
]

missing_cols = [c for c in required_base_cols if c not in raw.columns]
if missing_cols:
    raise ValueError(
        "step3_all_configurations_scored.csv is missing required columns: "
        + ", ".join(missing_cols)
    )

raw = coerce_numeric(
    raw,
    [
        "Rebalance Frequency",
        "l",
        "Cumulative Return",
        "Annualized Return",
        "Daily Sharpe",
        "Daily Max Drawdown",
        "Average Turnover",
        "Average Number of Stocks",
        "Rebalance Count",
        "Fallback Count",
    ],
)

# Keep only Experiment 1 population:
# - 27 core rows: Mapper / Global DBSCAN / PCA-KMeans × h × l
# - Benchmark rows: Equal Weight Universe and S&P 500 by h
allowed_methods = CORE_METHODS + BENCHMARK_METHODS
df = raw[raw["Method"].isin(allowed_methods)].copy()

# Normalize l to avoid 0.200000000001 issues
df["l"] = df["l"].astype(float)
df.loc[df["l"].notna(), "l"] = df.loc[df["l"].notna(), "l"].round(2)

# Keep exact intended population only
core_mask = (
    df["Method"].isin(CORE_METHODS)
    & df["Rebalance Frequency"].isin(REBALANCE_FREQUENCIES)
    & df["l"].isin(SELECTION_FRACTIONS)
)

benchmark_mask = (
    df["Method"].isin(BENCHMARK_METHODS)
    & df["Rebalance Frequency"].isin(REBALANCE_FREQUENCIES)
    & df["l"].isna()
)

df = df[core_mask | benchmark_mask].copy()

if "Strategy" not in df.columns:
    df["Strategy"] = df.apply(make_strategy_label, axis=1)
else:
    df["Strategy"] = df["Strategy"].fillna(df.apply(make_strategy_label, axis=1))

if "Rebalance Count" not in df.columns:
    df["Rebalance Count"] = np.nan

if "Fallback Count" not in df.columns:
    df["Fallback Count"] = np.nan

if "Notes" not in df.columns:
    df["Notes"] = ""

df["Rank Population"] = np.where(df["Method"].isin(CORE_METHODS), "Core grid", "Benchmark")


# ============================================================
# 4. VALIDATE POPULATION
# ============================================================

expected_core = pd.MultiIndex.from_product(
    [CORE_METHODS, REBALANCE_FREQUENCIES, SELECTION_FRACTIONS],
    names=["Method", "Rebalance Frequency", "l"]
).to_frame(index=False)

actual_core = df[df["Method"].isin(CORE_METHODS)][["Method", "Rebalance Frequency", "l"]].copy()
actual_core["Rebalance Frequency"] = actual_core["Rebalance Frequency"].astype(int)
actual_core["l"] = actual_core["l"].round(2)

merged_core = expected_core.merge(
    actual_core.drop_duplicates(),
    on=["Method", "Rebalance Frequency", "l"],
    how="left",
    indicator=True,
)

missing_core_rows = merged_core[merged_core["_merge"] == "left_only"].drop(columns=["_merge"])

expected_bench = pd.MultiIndex.from_product(
    [BENCHMARK_METHODS, REBALANCE_FREQUENCIES],
    names=["Method", "Rebalance Frequency"]
).to_frame(index=False)

actual_bench = df[df["Method"].isin(BENCHMARK_METHODS)][["Method", "Rebalance Frequency"]].copy()
actual_bench["Rebalance Frequency"] = actual_bench["Rebalance Frequency"].astype(int)

merged_bench = expected_bench.merge(
    actual_bench.drop_duplicates(),
    on=["Method", "Rebalance Frequency"],
    how="left",
    indicator=True,
)

missing_benchmark_rows = merged_bench[merged_bench["_merge"] == "left_only"].drop(columns=["_merge"])

dup_keys = (
    df.groupby(["Method", "Rebalance Frequency", "l"], dropna=False)
      .size()
      .reset_index(name="count")
)
duplicate_rows = dup_keys[dup_keys["count"] > 1].copy()

if not missing_core_rows.empty:
    raise ValueError(
        "Missing core grid rows. Cannot produce valid Experiment 1 ranking.\n"
        + missing_core_rows.to_string(index=False)
    )

if not duplicate_rows.empty:
    raise ValueError(
        "Duplicate Method/h/l rows detected. Cannot produce valid Experiment 1 ranking.\n"
        + duplicate_rows.to_string(index=False)
    )


# ============================================================
# 5. RECOMPUTE BALANCED SCORE
# ============================================================

score_df = df.copy()

# Higher is better:
# - Annualized Return
# - Daily Sharpe
# - Daily Max Drawdown, because less negative / higher drawdown value is better
#
# Lower is better:
# - Average Turnover
score_df["Annualized Return Score"] = normalized_rank_score(
    score_df["Annualized Return"],
    higher_is_better=True
)

score_df["Daily Sharpe Score"] = normalized_rank_score(
    score_df["Daily Sharpe"],
    higher_is_better=True
)

score_df["Daily Max Drawdown Score"] = normalized_rank_score(
    score_df["Daily Max Drawdown"],
    higher_is_better=True
)

score_df["Average Turnover Score"] = normalized_rank_score(
    score_df["Average Turnover"],
    higher_is_better=False
)

score_df["Balanced Score"] = (
    BALANCED_SCORE_WEIGHTS["Annualized Return Score"] * score_df["Annualized Return Score"]
    + BALANCED_SCORE_WEIGHTS["Daily Sharpe Score"] * score_df["Daily Sharpe Score"]
    + BALANCED_SCORE_WEIGHTS["Daily Max Drawdown Score"] * score_df["Daily Max Drawdown Score"]
    + BALANCED_SCORE_WEIGHTS["Average Turnover Score"] * score_df["Average Turnover Score"]
)

score_df["Balanced Score Rank"] = score_df["Balanced Score"].rank(
    ascending=False,
    method="min"
).astype(int)

score_df = score_df.sort_values(
    ["Balanced Score Rank", "Balanced Score", "Annualized Return", "Daily Sharpe"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

# Friendly l display
score_df["l Display"] = score_df["l"].map(
    lambda x: "" if pd.isna(x) else f"{int(round(float(x) * 100))}%"
)

# Add explicit rank columns for audit transparency
score_df["Annualized Return Rank"] = score_df["Annualized Return"].rank(
    ascending=False,
    method="min"
).astype(int)

score_df["Daily Sharpe Rank"] = score_df["Daily Sharpe"].rank(
    ascending=False,
    method="min"
).astype(int)

score_df["Daily Max Drawdown Rank"] = score_df["Daily Max Drawdown"].rank(
    ascending=False,
    method="min"
).astype(int)

score_df["Average Turnover Rank Lower Better"] = score_df["Average Turnover"].rank(
    ascending=True,
    method="min"
).astype(int)


# ============================================================
# 6. BUILD OUTPUT TABLES
# ============================================================

ranking_cols = [
    "Balanced Score Rank",
    "Strategy",
    "Rank Population",
    "Method",
    "Rebalance Frequency",
    "l",
    "l Display",
    "Cumulative Return",
    "Annualized Return",
    "Daily Sharpe",
    "Daily Max Drawdown",
    "Average Turnover",
    "Average Number of Stocks",
    "Rebalance Count",
    "Fallback Count",
    "Annualized Return Rank",
    "Daily Sharpe Rank",
    "Daily Max Drawdown Rank",
    "Average Turnover Rank Lower Better",
    "Annualized Return Score",
    "Daily Sharpe Score",
    "Daily Max Drawdown Score",
    "Average Turnover Score",
    "Balanced Score",
    "Notes",
]

ranking_cols = [c for c in ranking_cols if c in score_df.columns]
ranking = score_df[ranking_cols].copy()

# Best core configuration per method
core_scored = ranking[ranking["Method"].isin(CORE_METHODS)].copy()
best_core_by_method = (
    core_scored.sort_values(["Method", "Balanced Score Rank"])
    .groupby("Method", as_index=False)
    .head(1)
    .copy()
)
best_core_by_method.insert(0, "Top Configuration Section", "Best core configuration by method")

# Benchmark contextual rows
benchmark_rows = ranking[ranking["Method"].isin(BENCHMARK_METHODS)].copy()
benchmark_rows = benchmark_rows.sort_values(["Method", "Rebalance Frequency"])
benchmark_rows.insert(0, "Top Configuration Section", "Benchmark context row")

# Top 10 overall
top10 = ranking.head(10).copy()
top10.insert(0, "Top Configuration Section", "Top 10 overall by Balanced Score")

top_configurations = pd.concat(
    [best_core_by_method, benchmark_rows, top10],
    axis=0,
    ignore_index=True
)

# Remove exact duplicates if a row appears in multiple sections?
# Keep section labels, so do not deduplicate globally.


# ============================================================
# 7. WRITE REQUIRED OUTPUTS
# ============================================================

ranking_csv_path = tables_dir / "step4_full_balanced_score_ranking.csv"
ranking_md_path = tables_dir / "step4_full_balanced_score_ranking.md"
formula_json_path = tables_dir / "step4_balanced_score_formula.json"
top_configs_path = tables_dir / "step4_top_configurations.csv"
audit_md_path = tables_dir / "step4_balanced_score_audit.md"

safe_round_for_output(ranking).to_csv(ranking_csv_path, index=False)
safe_round_for_output(top_configurations).to_csv(top_configs_path, index=False)

ranking_md = "# Step 4 Experiment 1 — Full Balanced Score Ranking\n\n"
ranking_md += "This table recomputes Balanced Score from the saved Step 1–3 artifacts.\n\n"
ranking_md += dataframe_to_markdown(
    safe_round_for_output(
        ranking[
            [
                "Balanced Score Rank",
                "Strategy",
                "Rank Population",
                "Method",
                "Rebalance Frequency",
                "l Display",
                "Cumulative Return",
                "Annualized Return",
                "Daily Sharpe",
                "Daily Max Drawdown",
                "Average Turnover",
                "Average Number of Stocks",
                "Balanced Score",
            ]
        ]
    ),
    float_digits=6
)
ranking_md += "\n"
write_text(ranking_md_path, ranking_md)

formula = {
    "experiment": "Step 4 Experiment 1 - Full Balanced Score Ranking Table",
    "ranking_population": {
        "core_configurations": "3 methods x 3 rebalance frequencies x 3 selection fractions = 27 rows",
        "benchmark_rows": "Equal Weight Universe and S&P 500 for h in {21, 42, 63}, when available",
        "total_rows_expected_with_benchmarks": 33,
        "actual_rows_used": int(len(ranking)),
    },
    "balanced_score_formula": {
        "Balanced Score": "0.35 * Annualized Return Score + 0.35 * Daily Sharpe Score + 0.20 * Daily Max Drawdown Score + 0.10 * Average Turnover Score",
        "weights": BALANCED_SCORE_WEIGHTS,
    },
    "ranking_rules": {
        "Annualized Return": "Higher is better",
        "Daily Sharpe": "Higher is better",
        "Daily Max Drawdown": "Higher / less negative is better",
        "Average Turnover": "Lower is better",
    },
    "score_normalization": {
        "method": "Normalized rank score",
        "definition": "score = (N - rank + 1) / N, where best rank = 1 and best score = 1",
        "tie_method": "min rank",
        "note": "Average Turnover uses ascending rank because lower turnover is better.",
    },
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "input_files": {
        "experiment_input.zip_sha256": sha256_file(INPUT_ZIP),
        "plan.md_present": PLAN_MD.exists(),
        "plan.md_sha256": sha256_file(PLAN_MD) if PLAN_MD.exists() else "",
        "sp500_constituents_2020_start.csv_present_but_not_used_in_exp1": SP500_2020_FILE.exists(),
        "sp500_constituents_2020_start.csv_sha256": sha256_file(SP500_2020_FILE) if SP500_2020_FILE.exists() else "",
        "source_metrics_file_inside_zip": str(step3_all_path),
    },
}
with open(formula_json_path, "w", encoding="utf-8") as f:
    json.dump(formula, f, indent=2, ensure_ascii=False)

# Audit summary
top_row = ranking.iloc[0].to_dict()
top_core_row = core_scored.sort_values("Balanced Score Rank").iloc[0].to_dict()
pca_best = best_core_by_method[best_core_by_method["Method"] == "PCA-KMeans"].iloc[0].to_dict()

row_count_by_method = (
    ranking.groupby("Method")
    .size()
    .reset_index(name="row_count")
    .sort_values("Method")
)

core_row_count = int(ranking[ranking["Rank Population"] == "Core grid"].shape[0])
benchmark_row_count = int(ranking[ranking["Rank Population"] == "Benchmark"].shape[0])

pca_is_top_core = top_core_row["Method"] == "PCA-KMeans"

audit_text = "# Step 4 Experiment 1 — Balanced Score Audit\n\n"
audit_text += "## Input Source\n\n"
audit_text += f"- Source metrics file found inside zip: `{step3_all_path}`\n"
audit_text += f"- Generated at: `{datetime.now().isoformat(timespec='seconds')}`\n\n"

audit_text += "## Population Check\n\n"
audit_text += f"- Expected core rows: 27\n"
audit_text += f"- Actual core rows: {core_row_count}\n"
audit_text += f"- Expected benchmark rows if complete: 6\n"
audit_text += f"- Actual benchmark rows: {benchmark_row_count}\n"
audit_text += f"- Total rows ranked: {len(ranking)}\n\n"

audit_text += "### Row Count by Method\n\n"
audit_text += dataframe_to_markdown(row_count_by_method, float_digits=0)
audit_text += "\n\n"

audit_text += "## Missing Rows\n\n"
if missing_core_rows.empty:
    audit_text += "- Core grid: no missing rows.\n"
else:
    audit_text += "- Core grid missing rows:\n\n"
    audit_text += dataframe_to_markdown(missing_core_rows)
    audit_text += "\n"

if missing_benchmark_rows.empty:
    audit_text += "- Benchmark rows: no missing benchmark rows.\n\n"
else:
    audit_text += "- Benchmark rows missing:\n\n"
    audit_text += dataframe_to_markdown(missing_benchmark_rows)
    audit_text += "\n\n"

audit_text += "## Duplicate Rows\n\n"
if duplicate_rows.empty:
    audit_text += "- No duplicate Method / h / l rows detected.\n\n"
else:
    audit_text += dataframe_to_markdown(duplicate_rows)
    audit_text += "\n\n"

audit_text += "## Balanced Score Formula Used\n\n"
audit_text += "- Annualized Return Score weight: 0.35\n"
audit_text += "- Daily Sharpe Score weight: 0.35\n"
audit_text += "- Daily Max Drawdown Score weight: 0.20\n"
audit_text += "- Average Turnover Score weight: 0.10\n"
audit_text += "- For Annualized Return, Daily Sharpe, and Daily Max Drawdown: higher is better.\n"
audit_text += "- For Average Turnover: lower is better.\n"
audit_text += "- Score normalization: `(N - rank + 1) / N`, where best rank receives score 1.\n\n"

audit_text += "## Top Overall Row\n\n"
audit_text += f"- Strategy: `{top_row['Strategy']}`\n"
audit_text += f"- Method: `{top_row['Method']}`\n"
audit_text += f"- Balanced Score Rank: `{top_row['Balanced Score Rank']}`\n"
audit_text += f"- Balanced Score: `{top_row['Balanced Score']:.6f}`\n"
audit_text += f"- Annualized Return: `{top_row['Annualized Return']:.6f}`\n"
audit_text += f"- Daily Sharpe: `{top_row['Daily Sharpe']:.6f}`\n"
audit_text += f"- Daily Max Drawdown: `{top_row['Daily Max Drawdown']:.6f}`\n"
audit_text += f"- Average Turnover: `{top_row['Average Turnover']:.6f}`\n\n"

audit_text += "## Top Core Configuration\n\n"
audit_text += f"- Strategy: `{top_core_row['Strategy']}`\n"
audit_text += f"- Method: `{top_core_row['Method']}`\n"
audit_text += f"- Balanced Score Rank: `{top_core_row['Balanced Score Rank']}`\n"
audit_text += f"- Balanced Score: `{top_core_row['Balanced Score']:.6f}`\n\n"

audit_text += "## PCA-KMeans Claim Check\n\n"
if pca_is_top_core:
    audit_text += "- Result: PCA-KMeans is the highest-scoring core in-sample configuration in this ranking population.\n"
else:
    audit_text += "- Result: PCA-KMeans is NOT the highest-scoring core configuration in this recomputed ranking. Do not claim it as highest-scoring before investigating.\n"

audit_text += f"- Best PCA-KMeans strategy: `{pca_best['Strategy']}`\n"
audit_text += f"- Best PCA-KMeans Balanced Score Rank: `{pca_best['Balanced Score Rank']}`\n"
audit_text += f"- Best PCA-KMeans Balanced Score: `{pca_best['Balanced Score']:.6f}`\n\n"

audit_text += "## Files Generated\n\n"
for fname in REQUIRED_OUTPUTS:
    audit_text += f"- `{fname}`\n"

write_text(audit_md_path, audit_text)


# ============================================================
# 8. COPY REQUIRED FILES TO OUTPUT ROOT ALSO
# ============================================================
# This makes them easy to find without opening subfolders.

for p in [ranking_csv_path, ranking_md_path, formula_json_path, top_configs_path, audit_md_path]:
    shutil.copy2(p, OUTPUT_ROOT / p.name)


# ============================================================
# 9. ARTIFACT CHECK AND ZIP OUTPUT
# ============================================================

artifact_rows = []
for fname in REQUIRED_OUTPUTS:
    p1 = OUTPUT_ROOT / fname
    p2 = tables_dir / fname
    existing_path = p1 if p1.exists() else p2
    artifact_rows.append({
        "file_name": fname,
        "exists": existing_path.exists(),
        "size_bytes": existing_path.stat().st_size if existing_path.exists() else 0,
        "path": str(existing_path),
        "sha256": sha256_file(existing_path) if existing_path.exists() else "",
    })

artifact_check = pd.DataFrame(artifact_rows)
artifact_check_path = OUTPUT_ROOT / "step4_exp1_artifact_file_check.csv"
artifact_check.to_csv(artifact_check_path, index=False)

run_config = {
    "step": "Step 4",
    "experiment": "Experiment 1 - Full Balanced Score Ranking Table",
    "input_zip": str(INPUT_ZIP),
    "input_zip_sha256": sha256_file(INPUT_ZIP),
    "plan_md_present": PLAN_MD.exists(),
    "plan_md_sha256": sha256_file(PLAN_MD) if PLAN_MD.exists() else "",
    "source_metrics_file_inside_zip": str(step3_all_path),
    "core_methods": CORE_METHODS,
    "benchmark_methods": BENCHMARK_METHODS,
    "rebalance_frequencies": REBALANCE_FREQUENCIES,
    "selection_fractions": SELECTION_FRACTIONS,
    "expected_core_rows": 27,
    "actual_core_rows": core_row_count,
    "actual_benchmark_rows": benchmark_row_count,
    "actual_total_rows": int(len(ranking)),
    "balanced_score_weights": BALANCED_SCORE_WEIGHTS,
    "generated_at": datetime.now().isoformat(timespec="seconds"),
}
with open(metadata_dir / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2, ensure_ascii=False)

zip_output = Path("tda_step4_exp1_outputs.zip")
if zip_output.exists():
    zip_output.unlink()

shutil.make_archive(
    base_name=str(zip_output.with_suffix("")),
    format="zip",
    root_dir=OUTPUT_ROOT
)

print("DONE - Step 4 Experiment 1 outputs generated.")
print(f"Output folder: {OUTPUT_ROOT.resolve()}")
print(f"Output zip: {zip_output.resolve()}")
print()
print("Artifact check:")
display(artifact_check)

print()
print("Top 10 ranking preview:")
display(
    ranking[
        [
            "Balanced Score Rank",
            "Strategy",
            "Rank Population",
            "Method",
            "Rebalance Frequency",
            "l Display",
            "Annualized Return",
            "Daily Sharpe",
            "Daily Max Drawdown",
            "Average Turnover",
            "Balanced Score",
        ]
    ].head(10)
)

FileNotFoundError: Cannot find D:\Notebook\Professorship\research\experiment_input.zip. Please place experiment_input.zip in the same folder as this notebook.